# 04 — Create Genie Space

Creates (or updates) the J&J MedTech Sales Genie Space and wires it to the Delta tables and metric views created in notebooks 01–03.

Set `genie_space_id` to blank on first run. After creation the space ID will be printed — paste it back into the widget on re-runs to update rather than recreate.

In [ ]:
dbutils.widgets.text("catalog",        "medtech",  "Catalog")
dbutils.widgets.text("schema",         "sales",    "Schema")
dbutils.widgets.text("warehouse_id",   "",         "SQL Warehouse ID")
dbutils.widgets.text("genie_space_id", "",         "Genie Space ID (blank = create new)")

catalog        = dbutils.widgets.get("catalog").strip()
schema         = dbutils.widgets.get("schema").strip()
warehouse_id   = dbutils.widgets.get("warehouse_id").strip()
genie_space_id = dbutils.widgets.get("genie_space_id").strip()

if not warehouse_id:
    raise ValueError("warehouse_id is required")

print(f"Catalog:     {catalog}.{schema}")
print(f"Warehouse:   {warehouse_id}")
print(f"Genie space: {genie_space_id or '(will create new)'}")

In [ ]:
import json, requests
from databricks.sdk import WorkspaceClient

w       = WorkspaceClient()
host    = w.config.host.rstrip("/")
headers = {**dict(w.config.authenticate()), "Content-Type": "application/json"}

table_identifiers = [
    f"{catalog}.{schema}.account_targeting",
    f"{catalog}.{schema}.hcp_procedure_volume",
    f"{catalog}.{schema}.product_upgrades",
    f"{catalog}.{schema}.account_penetration",
    f"{catalog}.{schema}.market_exposure",
    f"{catalog}.{schema}.rolling_12_sales_metric",
    f"{catalog}.{schema}.total_max_market",
    f"{catalog}.{schema}.total_procedure_volume",
    f"{catalog}.{schema}.total_upgrade_opportunity",
    f"{catalog}.{schema}.yoy_procedure_growth",
]

sample_questions = [
    "Which account has the highest Opportunity?",
    "Who are the top 5 surgeons by procedure volume?",
    "Show opportunity by product line",
    "Which GPO has the highest opportunity?",
    "What is the total opportunity by target type?",
    "What is the average net cost for Alpha Series?",
    "Which accounts have declining penetration?",
    "Show top 10 accounts by rolling 12 sales",
    "What are the top upgrade opportunities?",
    "Show procedure volume by specialty",
]

description = (
    "J&J MedTech sales data advisor for analyzing HCP procedure volumes, product upgrade "
    "opportunities, and account targeting strategies. Covers 4 product lines (Alpha, Beta, "
    "Gamma, Delta) across 5 geographic areas. Target Types: Tier 1=Upgrade, "
    "Tier 2=Competitive, Tier 3=Market Expansion. Use account_targeting for opportunity "
    "and penetration questions. Use product_upgrades for product-level upgrade details. "
    "Use hcp_procedure_volume for surgeon/HCP procedure volume questions."
)

print(f"Tables: {len(table_identifiers)}")
print(f"Sample questions: {len(sample_questions)}")

In [ ]:
# Load serialized_space from genie_space.json (kept alongside this notebook in the workspace)
import os

notebook_dir = os.path.dirname(dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get())
genie_json_ws_path = f"/Workspace{notebook_dir}/genie_space.json"

serialised_space = ""
if os.path.exists(genie_json_ws_path):
    with open(genie_json_ws_path) as f:
        raw = json.load(f)
    serialised_space = raw.get("serialized_space", "")
    print(f"Loaded serialized_space from {genie_json_ws_path}")
else:
    print(f"genie_space.json not found at {genie_json_ws_path} — space will be created without pre-configured instructions")

In [ ]:
payload = {
    "warehouse_id":      warehouse_id,
    "title":             "J&J MedTech Sales Genie Space",
    "description":       description,
    "table_identifiers": table_identifiers,
    "sample_questions":  sample_questions,
}
if serialised_space:
    payload["serialized_space"] = serialised_space

final_space_id = None

if genie_space_id:
    print(f"Updating Genie space {genie_space_id} ...")
    resp = requests.patch(
        f"{host}/api/2.0/genie/spaces/{genie_space_id}",
        headers=headers, json=payload
    )
    if resp.status_code == 200:
        final_space_id = genie_space_id
        print(f"Updated: {final_space_id}")
    else:
        print(f"Update failed ({resp.status_code}) — creating a new space instead.")

if not final_space_id:
    print("Creating new Genie space ...")
    resp = requests.post(f"{host}/api/2.0/genie/spaces", headers=headers, json=payload)
    if resp.status_code not in (200, 201):
        raise Exception(f"Failed ({resp.status_code}): {resp.text[:400]}")
    data = resp.json()
    final_space_id = data.get("space_id") or data.get("id")
    print(f"Created: {final_space_id}")

print()
print("=" * 60)
print(f"  GENIE SPACE ID: {final_space_id}")
print(f"  Paste this into the genie_space_id widget on re-runs.")
print("=" * 60)